In [1]:
!pip install transformers datasets accelerate -q

In [2]:
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Libraries imported successfully!")
print(f"Using device: {device}")

✅ Libraries imported successfully!
Using device: cuda


In [3]:
training_text = """
Artificial intelligence is transforming every industry in the world.
Machine learning models are trained on large amounts of data.
Deep learning uses neural networks with multiple hidden layers.
Natural language processing helps computers understand human language.
GPT-2 is a transformer-based language model developed by OpenAI.
Fine-tuning allows us to adapt pre-trained models to specific tasks.
Python is the most popular programming language for AI and ML.
Neural networks learn by adjusting weights through backpropagation.
Data preprocessing is an important step before training any model.
Transfer learning saves time by reusing knowledge from pre-trained models.
Reinforcement learning trains agents by rewarding correct behavior.
Computer vision enables machines to interpret and understand images.
Speech recognition converts spoken language into written text.
Chatbots use natural language processing to simulate conversations.
AI is being used in healthcare, finance, education, and many more fields.
"""

# Save to file
with open("train.txt", "w") as f:
    f.write(training_text * 20)  # repeat to create enough training data

print("✅ Dataset created!")
print(f"Total characters: {len(training_text * 20)}")

✅ Dataset created!
Total characters: 20320


In [4]:
print("Loading GPT-2 model and tokenizer...")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # fix padding issue

model = GPT2LMHeadModel.from_pretrained("gpt2")
model = model.to(device)

print(f"✅ Model loaded!")
print(f"Total parameters: {model.num_parameters():,}")

Loading GPT-2 model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded!
Total parameters: 124,439,808


In [5]:
class TextDatasetCustom(Dataset):
    def __init__(self, tokenizer, file_path, block_size=64):
        with open(file_path, "r") as f:
            text = f.read()

        tokenized = tokenizer(
            text,
            return_tensors="pt",
            truncation=False,
            padding=False
        )["input_ids"][0]

        self.examples = []
        for i in range(0, len(tokenized) - block_size, block_size):
            self.examples.append(tokenized[i : i + block_size])

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return {
            "input_ids": self.examples[idx],
            "labels":    self.examples[idx]
        }

train_dataset = TextDatasetCustom(
    tokenizer=tokenizer,
    file_path="train.txt",
    block_size=64
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(f"✅ Dataset ready!")
print(f"Number of training samples: {len(train_dataset)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3940 > 1024). Running this sequence through the model will result in indexing errors


✅ Dataset ready!
Number of training samples: 61


In [6]:
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    save_steps=50,
    logging_steps=10,
    save_total_limit=2,
    prediction_loss_only=True,
    fp16=torch.cuda.is_available(),
)

print("✅ Training settings configured!")
print(f"Epochs     : {training_args.num_train_epochs}")
print(f"Batch size : {training_args.per_device_train_batch_size}")
print(f"FP16 mode  : {training_args.fp16}")

✅ Training settings configured!
Epochs     : 5
Batch size : 4
FP16 mode  : True


In [7]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

print("🚀 Starting training... please wait")
trainer.train()
print("✅ Training complete!")

🚀 Starting training... please wait


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.200016
20,1.426720
30,0.584283
40,0.306457
50,0.202991
60,0.165430
70,0.130788
80,0.136293


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training complete!


In [8]:
model.save_pretrained("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

print("✅ Model saved to ./gpt2-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ./gpt2-finetuned


In [9]:
import math

# Calculate perplexity from training loss directly
logs = trainer.state.log_history

# Get last recorded loss
train_loss = [x['loss'] for x in logs if 'loss' in x]
last_loss = train_loss[-1]
perplexity = math.exp(last_loss)

print(f"\n📊 EVALUATION RESULTS")
print(f"Train Loss  : {last_loss:.4f}")
print(f"Perplexity  : {perplexity:.2f}")

if perplexity < 10:
    print("Rating      : 🟢 Excellent")
elif perplexity < 30:
    print("Rating      : 🟡 Good")
else:
    print("Rating      : 🔴 Needs more training")


📊 EVALUATION RESULTS
Train Loss  : 0.1363
Perplexity  : 1.15
Rating      : 🟢 Excellent


In [10]:
from transformers import pipeline
import torch

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

prompts = [
    "Artificial intelligence",
    "Machine learning models",
    "Deep learning",
    "Python is"
]

print("=" * 50)
print("       GENERATED TEXT OUTPUTS")
print("=" * 50)

for prompt in prompts:
    output = generator(
        prompt,
        max_length=60,
        num_return_sequences=1,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    print(f"\n📝 Prompt : '{prompt}'")
    print(f"🤖 Output : {output[0]['generated_text']}")
    print("-" * 50)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'pad_token_id', 'num_return_sequences', 'do_sample', 'max_length', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


       GENERATED TEXT OUTPUTS


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Prompt : 'Artificial intelligence'
🤖 Output : Artificial intelligence is transforming every industry in the world.
Machine learning models are trained on large amounts of data.
Deep learning uses neural networks with multiple hidden layers.
Natural language processing helps computers understand human language.
GPT-2 is a transformer-based language model developed by OpenAI.
Fine-tuning allows us to adapt pre-trained models to specific tasks.
Python is the most popular programming language for AI and ML.
Neural networks learn by adjusting weights through backpropagation.
Data preprocessing is an important step before training any model.
Transfer learning saves time by reusing knowledge from pre-trained models.
Reinforcement learning trains agents by rewarding correct behavior.
Computer vision enables machines to interpret and understand images.
Speech recognition converts spoken language into written text.
AI is being used in healthcare, finance, education, and many more fields.

Mac

[transformers] Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Prompt : 'Machine learning models'
🤖 Output : Machine learning models are trained on large amounts of data.
Deep learning uses neural networks with multiple hidden layers.
Natural language processing helps computers understand human language.
GPT-2 is a transformer-based language model developed by OpenAI.
Fine-tuning allows us to adapt pre-trained models to specific tasks.
Python is the most popular programming language for AI and ML.
Neural networks learn by adjusting weights through backpropagation.
Data preprocessing is an important step before training any model.
Transfer learning saves time by reusing knowledge from pre-trained models.
Reinforcement learning trains agents by rewarding correct behavior.
Data preprocessing is an important step before training any model.
Transfer learning saves time by reusing knowledge from pre-trained models.
Python is the most popular programming language for AI and ML.
Fine-tuning allows us to adapt pre-trained models to specific tasks.
Data 

[transformers] Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📝 Prompt : 'Deep learning'
🤖 Output : Deep learning models use pre-imported pre-trained models.
Python is the most popular programming language for AI and ML.
Neural networks learn by adjusting weights through backpropagation.
Data preprocessing is an important step before training any model.
Transfer learning saves time by reusing knowledge from pre-trained models.
Reinforcement learning trains agents by rewarding correct behavior.
Computer vision enables machines to interpret and understand images.
Speech recognition converts spoken language into written text.
Chatbots use natural language processing to simulate conversations.
Chatbots use natural language processing to simulate conversations.
AI is being used in healthcare, finance, education, and many more fields.

Artificial intelligence is transforming every industry in the world.
Machine learning models are trained on large amounts of data.
Deep learning uses neural networks with multiple hidden layers deep.
Natural language pr